In [1]:
import PyPDF2
import re
import csv

import numpy as np

In [ ]:
def extract_archery_results(pdf_path):
    # Lire le PDF
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"

    #print("debug text : ",text)
    # Initialiser les données
    archers = []
    
    # Séparer le texte en lignes
    lines = text.split('\n')

    # Variables pour stocker les informations de l'archer actuel
    current_archer = None

    in_scores = False
    #cpt = 0
    for line in lines:
        #cpt+=1
        line = line.strip()


        # Détecter le nom de l'archer
        name_det_str = "Archer: "
        if name_det_str in line: ### startswith 
            #if cpt == 5620+1:
             #   print(line)
              #  print(line[line.find(name_det_str)+len(name_det_str):])
            current_archer = {
                "Nom": line[line.find(name_det_str)+len(name_det_str):], #.strip(),
                "Id_Club": None,
                "Club": None,
                "Catégorie": None,
                "Scores": []
            }

            #if line.replace("Archer:", "").strip()=="Randriantsara Lai-Tsan": #
             #   for ii in range(80):
              #      print(lines[cpt-2+ii], cpt-2+ii)
            
                #print(current_archer)
            if current_archer:
                archers.append(current_archer)
                #print("debug current_archer :",current_archer)
            continue

        # Détecter le club et la catégorie
     
        if line.startswith("Clubs / Pays:"):
            if current_archer:
                parts = line.replace("Clubs / Pays:", "").strip().split()
                # Extraire le club (tout ce qui précède la catégorie)
                club_parts = []
                for part in parts:
                    if part.isdigit() or part == "-":
                        club_parts.append(part)
                    else:
                        break
                id_club = " ".join(club_parts).strip()
                id_club = ''.join(re.sub('[-]', '', id_club).split())
                current_archer["Id_Club"] = id_club
                # La catégorie est le reste de la ligne
                nom_club = " ".join(parts[len(club_parts):]).strip()
                cible = re.findall(r'[1-9][ABCD]|[1-9][0-9][ABCD]', nom_club)
                nom_club = re.sub(cible[0], '', nom_club)
                current_archer["Club"] = nom_club
            #print("debug Id_Club :",current_archer["Id_Club"])
            #print("debug Club :",current_archer["Club"])   
            continue

        # Détecter la catégorie (CO S1H, etc.)
        if line.startswith("CO"):
            current_archer["Catégorie"] = line#.strip()
            #print("debug Catégorie CO :",current_archer["Catégorie"])
            continue

        # Détecter la catégorie (CL S1H, etc.)
        if line.startswith("CL"):
            current_archer["Catégorie"] = line#.strip()
            #print("debug Catégorie CL :",current_archer["Catégorie"])
            continue

        # Détecter les scores après "Somme Tot. 10+XX"
        if "Somme Tot. 10+XX" in line:
            in_scores = True
            score_lines = []
            continue

        # Extraire les 12 lignes de scores
        if in_scores:
            score_lines.append(line)
            if len(score_lines) == 12:
                # Traiter les 12 lignes de scores
                score = ['','','','','','']
                for i in range(0, 12, 2):
                    # Ligne impaire : numéro de volée + 3 scores
                    impair_line = ''.join(score_lines[i].split())
                    id_volee = impair_line[0]
                    score[0:3] = re.findall(r'10|[1-9XM]', impair_line[1:-1])[:3] 

                    # Ligne paire : 3 scores
                    pair_line = ''.join(score_lines[i + 1].split())
                    score[3:-1] = re.findall(r'10|[1-9XM]', pair_line[0:-1])[:3] 

                    # Ajouter les scores pour cette volée
                    current_archer["Scores"].append({
                        "Volée": id_volee,
                        "Score1": score[0],
                        "Score2": score[1],
                        "Score3": score[2],
                        "Score4": score[3],
                        "Score5": score[4],
                        "Score6": score[5]
                    })
                in_scores = False
            continue


    # Ajouter le dernier archer
    if current_archer:
        archers.append(current_archer)

    return archers

In [56]:
# Exemple d'utilisation
path_pdf  = "./data_ianseo/TAE_2026_D1_etape2_qualif_FCL_HCO.pdf"
path_save = "./data_ianseo/"
results = extract_archery_results(path_pdf)


# Préparer les noms de colonnes pour le CSV
header = ["Nom", "Id_Club", "Club", "Catégorie"]
for volee in range(1, 7):
    for score_num in range(1, 7):
        header.append(f"Volee_{volee}_Score_{score_num}")

# Écrire dans le fichier CSV
with open(path_save+"resultats_tir_arc.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)  # Écrire l'en-tête

    # Afficher les résultats
    for archer in results:
    # Préparer les données pour le CSV
        row = [
            archer["Nom"],
            archer["Id_Club"],
            archer["Club"],
            archer["Catégorie"]
        ]
        for volee_data in archer["Scores"]:
            for score_key in ["Score1", "Score2", "Score3", "Score4", "Score5", "Score6"]:
                row.append(volee_data[score_key])


        writer.writerow(row)    # Écrire les données de l'archer



Total 312 94Archer: Randriantsara Lai-Tsan
Randriantsara Lai-Tsan


In [16]:
results?

Type:        list
String form: [{'Nom': 'Piron Benjamin', 'Id_Club': '0895293', 'Club': 'Sarcelles', 'Catégorie': 'CO S1H', 'Sco <...>  Salma', 'Id_Club': '1313090', 'Club': 'St Martin De Crau', 'Catégorie': 'CL S1F', 'Scores': []}]
Length:      296
Docstring:  
Built-in mutable sequence.

If no argument is given, the constructor creates a new empty list.
The argument must be an iterable if specified.